[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/projection/projection.ipynb)

# Cameras and Shadows by Join and Meet

This example models pinhole cameras, cast shadows, and two-view stereo correspondence in three-dimensional projective space through one incidence construction. Joining a world point with an optical centre forms a sight ray, and meeting that ray with an image plane places the point's perspective image on the screen. Replacing the camera and screen with a lamp and the ground casts shadows, a distant sun is modelled as an ideal point at infinity whose rays run parallel, and posing the same camera twice ties the two views together along epipolar lines.

In projective geometric algebra the line through two points is their regressive join `&`, and the intersection with a plane is their exterior meet `^`. The pinhole camera `(centre & Point) ^ screen` is a linear map from points to points that treats a finite lamp and a directional sun identically. Evaluating `(Point & corner) ^ ground` as a map on the light sweeps a fixed corner's shadow across an entire light path at once; `(centre & Line) ^ screen` images lines directly onto epipolar lines; and `(centre_1 & Point) ^ (centre_2 & Point)` forms the bilinear two-view correspondence map that vanishes whenever two sight rays intersect.

In the homogeneous matrix notation of projective vision, a calibrated camera with rotation $R$ and optical centre $\mathbf C$ is assembled as the $3 \times 4$ block matrix $P = K R [I_3 \mid -\mathbf C]$, planar shadow projection under a homogeneous light $\mathbf L$ onto a plane $\boldsymbol\pi$ requires the $4 \times 4$ rank-three matrix $S = (\boldsymbol\pi^{\mathsf T}\mathbf L)\,I_4 - \mathbf L\boldsymbol\pi^{\mathsf T}$, Plücker line projection needs a separate $3 \times 6$ line camera matrix, and two-view incidence is written via a skew cross-product matrix $[\mathbf t]_\times$ as $\mathbf x_2^{\mathsf T} K_2^{-\mathsf T} [\mathbf t]_\times R\, K_1^{-1} \mathbf x_1 = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure
from examples.geometry.projection import render, scenarios

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                                                 # Antivector-
Line = ga.gatype.antibivector()                                                # Antibivector-
Plane = ga.gatype.vector()                                                     # Vector+
Motor = ga.gatype.rotor()                                                      # Rotor+
Pseudoscalar = ga.gatype.pseudoscalar()                                        # Pseudoscalar+
Correspondence = ga.gatype((Pseudoscalar, Point, Point))                       # Pseudoscalar+ <- (Point-, Point-)

## Pinhole camera map

A camera at the origin looking along the positive depth axis onto the screen `z - w` evaluates `(origin & Point) ^ screen`. Its coefficient table carries perspective division directly in projective coordinates because the image point's weight equals the world point's depth.

In [ ]:
origin = mv.w.dual()                                                           # [] Point-
screen = mv.z - mv.w                                                           # [] Plane+
camera = (origin & Point) ^ screen                                             # [] Point- <- Point-
print("the camera, a map from points to their images:\n", camera.coefficient_table())

## Cast shadows from finite and ideal lights

A cube hangs above the ground plane, lit simultaneously by a nearby point lamp and by a directional sun at infinity. Each light's shadow operator has the same form as the pinhole camera with the light as centre and the ground as screen, while binding the corner instead turns the expression into a linear map of the light's position along a circular path.

In [ ]:
cube_edges = scenarios.cube_edges                                              # [12, 2]
body_side, body_height = scenarios.body_side, scenarios.body_height
point_light = mv.point(scenarios.point_light_coords)                           # [] Point-
sun = mv.direction(scenarios.sun_direction)                                    # [] Point-
light_path_centre, n_light_path = scenarios.light_path_centre, scenarios.n_light_path

In [ ]:
def circle(n_points: int) -> Point:
    """Evenly spaced rotor turns in the xy plane sweep `[n_points] Point-` along the unit circle about the origin."""
    angles = np.linspace(0.0, 2.0 * np.pi, n_points, endpoint=False)
    turns = (mv.xy * (-angles / 2)).exp()                                      # [n_points] Motor+
    return turns >> mv.point([1, 0, 0])


def cube(side: float) -> Point:
    """An axis-aligned cube of edge length `side` centred on the origin has eight corners (`[8] Point-`)."""
    corners = np.array([[x, y, z] for x in (-1, 1) for y in (-1, 1) for z in (-1, 1)])  # [8, 3]
    return mv.point(corners * (side / 2.0))

In [ ]:
def shadows(
    body: Point, ground: Plane, point_light: Point, sun: Point,
    corner: Point, light_path: Point,
) -> tuple[Point, Point, Point]:
    """Joining points with a light and meeting the ground casts `body`'s spotlight and sunlight shadows (`[...] Point-`) and `corner`'s shadow trail along `light_path`."""
    # Join the point with the light into a ray, then meet the ray with the ground; a finite point light
    # and an ideal direction at infinity evaluate the same linear map Point- <- Point-.
    spotlight = (point_light & Point) ^ ground                                 # [] Point- <- Point-
    sunlight = (sun & Point) ^ ground                                          # [] Point- <- Point-

    # Fixing the corner instead makes its shadow a linear map of the light's position, casting an entire
    # path of light positions in one call.
    shadow_of_corner = (Point & corner) ^ ground                               # [] Point- <- Point-

    return spotlight(body), sunlight(body), shadow_of_corner(light_path)


body_lift = (mv.zw * (body_height / 2)).exp()                                  # [] Motor+
body = body_lift >> cube(body_side)                                            # [8] Point-
ground = mv.z                                                                  # [] Plane+
path_shift = ((stack((mv.xw, mv.yw, mv.zw)) * light_path_centre).sum(axis=0) / 2).exp()  # [] Motor+
light_path = path_shift >> circle(n_light_path)                                # [n_light_path] Point-
cast = shadows(body, ground, point_light, sun, body[7], light_path)

## Stereo views and epipolar geometry

Posing two copies of the camera on either side of a second cube conjugates the camera map by each rig motor, pulling world points into the rig frame and pushing screen images back out. Imaging the first optical centre in the second camera locates the epipole, applying the line camera to the first view's sight rays draws the epipolar lines directly, and wedging the two sight rays forms the bilinear correspondence constraint.

In [ ]:
rig_shift, rig_turn = scenarios.rig_shift, scenarios.rig_turn
subject_side, subject_depth = scenarios.subject_side, scenarios.subject_depth

In [ ]:
def stereo(
    subject: Point, centre: Point, screen: Plane,
    rig_1: Motor, rig_2: Motor,
) -> tuple[Point, Point, Point, Line, Correspondence]:
    """Two camera poses `rig_1` and `rig_2` image `subject` (`[...] Point-`) and determine the second view's epipole, epipolar lines (`[...] Line-`), and two-view correspondence form."""
    # Joining with the optical centre and meeting the screen gives both the point camera and the line camera.
    camera = (centre & Point) ^ screen                                         # [] Point- <- Point-
    line_camera = (centre & Line) ^ screen                                     # [] Line- <- Line-

    # Conjugating each camera map by its pose motor pulls world elements into the rig frame and pushes
    # screen images out, keeping the image of a join equal to the join of the images.
    camera_1 = rig_1 >> camera(rig_1 << Point)                                 # [] Point- <- Point-
    camera_2 = rig_2 >> camera(rig_2 << Point)                                 # [] Point- <- Point-
    line_camera_2 = rig_2 >> line_camera(rig_2 << Line)                        # [] Line- <- Line-
    centre_1 = rig_1 >> centre                                                 # [] Point-
    centre_2 = rig_2 >> centre                                                 # [] Point-

    image_1 = camera_1(subject)                                                # [...] Point-
    image_2 = camera_2(subject)                                                # [...] Point-

    # The epipole images the other optical centre, and the line camera maps each first-view sight ray
    # directly onto its epipolar line on the second screen.
    epipole_2 = camera_2(centre_1)                                             # [] Point-
    epipolar_lines_2 = line_camera_2(centre_1 & image_1)                       # [...] Line-

    # Two image points correspond when their sight rays meet, which makes the wedge of those rays vanish
    # as a bilinear form on both image points.
    correspondence = (centre_1 & Point) ^ (centre_2 & Point)                   # [] Pseudoscalar+ <- (Point-, Point-)

    return image_1, image_2, epipole_2, epipolar_lines_2, correspondence


rig_1 = (mv.xw * -rig_shift).exp() * (mv.xz * +rig_turn).exp()                 # [] Motor+
rig_2 = (mv.xw * +rig_shift).exp() * (mv.xz * -rig_turn).exp()                 # [] Motor+
subject_lift = (mv.zw * (subject_depth / 2)).exp()                             # [] Motor+
subject = subject_lift >> cube(subject_side)                                   # [8] Point-

views = stereo(subject, origin, screen, rig_1, rig_2)
image_1, image_2, epipole_2, epipolar_lines_2, correspondence = views
save_figure(render.projection_figure(body, cube_edges, cast, point_light, sun, views, rig_1, rig_2), "projection");

The left panel shows the blue cube floating above the ground, its yellow perspective shadow cast by the amber point lamp, its parallel purple shadow cast along the sun arrow, and the yellow bead ring traced by one cube corner as the lamp circles overhead. The middle and right panels show the two camera screens, where every red corner marker in the second view lies on the red epipolar line projected from its matching sight ray in the first view.

In [ ]:
# checks
# Every shadow lies on the ground; corresponding images annihilate the correspondence form, the epipole lies in
# its kernel, and each epipolar line passes through its image.
for shadow in cast:
    assert ((ground & shadow).abs() < 1e-12).all()
assert (correspondence(image_1, image_2).dual().abs() < 1e-12).all()
assert (correspondence(image_1, epipole_2).dual().abs() < 1e-12).all()
assert ((epipolar_lines_2 & image_2).norm() < 1e-12).all()